In [ ]:
#use "../lib/nb.ml";;
load_css ()

# An EBNF based Parser for Arithmetic Expressions

In this notebook we implement an <span style="font-variant:small-caps;">Ebnf</span> recursive-descend parser for arithmetic expressions.  This parser implements the following <span style="font-variant:small-caps;">Ebnf</span> grammar:
$$
  \begin{eqnarray*}
  \mathrm{expr}    & \rightarrow & \mathrm{product}\;\;\bigl((\texttt{'+'}\;|\;\texttt{'-'})\;\; \mathrm{product}\bigr)^* \\[0.2cm]
  \mathrm{product} & \rightarrow & \mathrm{factor} \;\;\bigl((\texttt{'*'}\;|\;\texttt{'/'})\;\; \mathrm{factor}\bigr)^*  \\[0.2cm]   
  \mathrm{factor}  & \rightarrow & \texttt{'('} \;\;\mathrm{expr} \;\;\texttt{')'}                             \\
                   & \mid        & \texttt{NUMBER} 
  \end{eqnarray*}
$$

## Type Checking

The functions in this notebook carry *type annotations*.  In OCaml, these annotations are checked by the
compiler before a cell is executed, so there is no need for a separate type checker like *basedpyright*.
A cell that contains a type error is not executed at all.

## The Scanner

We use the module `Re` from the file `../lib/regex.ml`, which offers the functions of Python's module `re` on top of the library PCRE2.

In [ ]:
load_lib "regex.ml"

The function `tokenize` receives a string `s` as argument and returns a list of tokens.
The string `s` is supposed to represent an arithmetical expression. 

**Note:** 
 - We need to set the flag `VERBOSE` in our call of the function `Re.findall`
   below because otherwise we are not able to format the regular expression `lexSpec` the way 
   we have done it.  Furthermore, we wouldn't have been able to add comments inside the regular expression.
 - Since the regular expression does not allow white space, `tokenList` will contain lots of 
   empty strings.  These have to be removed.

In [ ]:
let tokenize (s : string) : string list =
  let lexSpec = {|[a-z][0-9]*   |  # variables
                  [1-9][0-9]*|0 |  # numbers
                  [-+*/()]      |  # arithmetical operators and parentheses
               |}
  in
  let tokenList = Re.findall ~flags:Re.[VERBOSE] lexSpec s in
  List.filter (fun t -> t <> "") tokenList

In [ ]:
tokenize "12 * x + y1 * 4 / 6 - z3"

## Implementing the Recursive Descend Parser

The type `ast` represents an *abstract syntax tree*.  In Python, it is the type `int | str | tuple[str, AST, AST]`:
a number, a variable, or a triple consisting of an operator and two subtrees.  In OCaml, these three cases are
the constructors `Num`, `Var`, and `BinOp`, and the operators are the constructors of the type `op`.

**Type safety:** In Python, the first component of a triple can be any string.  In OCaml, it is one of the four
operators `Plus`, `Minus`, `Times`, and `Divide`, so a misspelled operator is rejected by the compiler.  The function
`op_of_string` converts a token into an operator.

In [ ]:
type op = Plus | Minus | Times | Divide

type ast =
  | Num   of int
  | Var   of string
  | BinOp of op * ast * ast

let op_of_string (s : string) : op =
  match s with
  | "+" -> Plus | "-" -> Minus | "*" -> Times | "/" -> Divide
  | _ -> invalid_arg ("op_of_string: " ^ s)

The function `parse` takes a string `s` as input and parses this string according to the recursive grammar
shown above.

The parsing functions call each other: `parse` calls `parseExpr`, which is only defined below, and
`parseFactor` calls `parseExpr` again.  In order to keep every function in a cell of its own, we use *forward
references*: the next cell creates references, i.e. mutable boxes, that will later hold the parsing functions.
`!parseExpr_ref` is the function that is currently stored in the box `parseExpr_ref`.  After a function has
been defined, it is stored in its box with the operator `:=`.

In [ ]:
let not_yet_defined _ = failwith "not yet defined"

let parseExpr_ref    : (string list -> ast * string list) ref = ref not_yet_defined
let parseProduct_ref : (string list -> ast * string list) ref = ref not_yet_defined
let parseFactor_ref  : (string list -> ast * string list) ref = ref not_yet_defined

let show_tokens (tl : string list) : string = "[" ^ String.concat ", " (List.map repr tl) ^ "]"

In [ ]:
let parse (s : string) : ast =
  let tl = tokenize s in
  match !parseExpr_ref tl with
  | result, [] -> result
  | _, rest -> failwith (Printf.sprintf "Parse Error: could not parse %s, Rest = %s" (show_tokens tl) (show_tokens rest))

The function `parseExpr tl` takes a list of tokens `tl` and tries to parse an expresssion according to the following
<span style="font-variant:small-caps;">Ebnf</span> grammar rule:
$$ \mathrm{expr} \;\rightarrow\; \mathrm{product}\;\;\bigl((\texttt{'+'}\;|\;\texttt{'-'})\;\; \mathrm{product}\bigr)^* $$
It returns an *abstract syntax tree* representing the expression and a list of all the tokens that have not been consumed during parsing.
The `while` loop of Python is replaced by the recursive function `loop`.

In [ ]:
let parseExpr (tl : string list) : ast * string list =
  let rec loop result rest =
    match rest with
    | operator :: (_ :: _ as rest) when operator = "+" || operator = "-" ->
        let arg, rest = !parseProduct_ref rest in
        loop (BinOp (op_of_string operator, result, arg)) rest
    | _ -> result, rest
  in
  let result, rest = !parseProduct_ref tl in
  loop result rest
;;
parseExpr_ref := parseExpr

The function `parseProduct tl` takes a list of tokens `tl` and tries to parse a product according to the following
<span style="font-variant:small-caps;">Ebnf</span> grammar rule:
$$ \mathrm{product} \;\rightarrow\; \mathrm{factor} \;\;\bigl((\texttt{'*'}\;|\;\texttt{'/'})\;\; \mathrm{factor}\bigr)^*  $$
It returns the value of the product and a list of all the tokens that have not been consumed during parsing.

In [ ]:
let parseProduct (tl : string list) : ast * string list =
  let rec loop result rest =
    match rest with
    | operator :: (_ :: _ as rest) when operator = "*" || operator = "/" ->
        let arg, rest = !parseFactor_ref rest in
        loop (BinOp (op_of_string operator, result, arg)) rest
    | _ -> result, rest
  in
  let result, rest = !parseFactor_ref tl in
  loop result rest
;;
parseProduct_ref := parseProduct

The function `parseFactor` implements the following grammar rules:
$$
  \begin{eqnarray*}
  \mathrm{factor}      & \;\rightarrow\; & \texttt{'('} \;\;\mathrm{expr} \;\;\texttt{')'}    \\
                       & \;\mid          & \;\texttt{NUMBER}                                  \\
                       & \;\mid          & \;\texttt{VARIABLE}
  \end{eqnarray*}
$$

It takes one argument:
- `tl` is the list of tokens that still need to be consumed.

It returns a pair of the form `(value, rest)` where
- `value` is the result of evaluating the arithmetical expression
  that is represented by `tl` and
- `rest` is a list of those tokens that have not been consumed while trying to parse a factor.

In [ ]:
let parseFactor (tl : string list) : ast * string list =
  match tl with
  | "(" :: rest -> (
      match parseExpr rest with
      | expr, ")" :: rest -> expr, rest
      | _, t :: _ -> failwith (Printf.sprintf "ERROR: ')' expected, got %s" t)
      | _, [] -> failwith "ERROR: ')' expected, got end of input")
  | t :: rest when '0' <= t.[0] && t.[0] <= '9' -> Num (int_of_string t), rest
  | t :: rest -> Var t, rest
  | [] -> failwith "ERROR: unexpected end of input"
;;
parseFactor_ref := parseFactor

## Testing

In order to draw an abstract syntax tree, we load the notebook `AST2Dot.ipynb`, which also loads the module
`Graphviz`.  The tree is converted into a nested tuple of the type `nested`, which is defined in this notebook.

In [ ]:
run_notebook "AST2Dot.ipynb"

In [ ]:
let rec to_nested (t : ast) : nested =
  match t with
  | Num n -> Int n
  | Var v -> Str v
  | BinOp (op, lhs, rhs) ->
      let label = match op with Plus -> "+" | Minus -> "-" | Times -> "*" | Divide -> "/" in
      Tuple (label, [ to_nested lhs; to_nested rhs ])

**Note:** The function `run_notebook` executes all cells of a notebook, just like the magic command `%run`.  The
OCaml compiler knows all definitions of the loaded notebook, so every cell of this notebook is type checked completely.

In [ ]:
let test (s : string) : Graphviz.t =
  let tree = parse s in
  tuple2dot (to_nested tree)

In [ ]:
parse "12 * y * x + 14 * z / 6 - x"

In [ ]:
Graphviz.display (test "2 * x + y * y - z / (x * x + y * y) - 3")